# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [ ]:
import json
import requests
import pandas as pd
import time
from datetime import date

with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']
BASE_URL = 'https://content.guardianapis.com'

# generate (first_day, last_day) pairs for each month
month_starts = pd.date_range("2020-01-01", "2026-03-31", freq="MS")
date_ranges = [
    (d.strftime("%Y-%m-%d"), (d + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d"))
    for d in month_starts
]

ARTICLES_PER_MONTH = 50
all_results = []



for from_date, to_date in date_ranges:
    for page in range(1, (ARTICLES_PER_MONTH // 50) + 1):
        parameters = {
            "api-key": API_KEY,
            "q": "oil OR natural gas OR gold OR OPEC OR energy crisis",
            "page-size": ARTICLES_PER_MONTH,
            "page": page,
            "show-fields": "bodyText",
            "from-date": from_date,
            "to-date": to_date,
            "order-by": "relevance"
        }
        response = requests.get(f"{BASE_URL}/search", params=parameters)

        data = response.json()['response']

        for article in data['results']:
            all_results.append({
                'date': article.get('webPublicationDate'),
                'section': article.get('sectionName'),
                'title': article.get('webTitle'),
            })

        if page >= data['pages']:
            break
        time.sleep(0.5)


print(f"Done.  Total: {len(all_results)}")



In [ ]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
# all futures
commodities = {
    "Gold":        "GC=F",   
    "Brent Oil":   "BZ=F",   # brent crude oil
    "Natural Gas": "NG=F",   
    "Wheat":       "ZW=F",   
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2020-01-01",
    end="2025-12-31",
    interval="1wk"       # 1d for daily, 1wk for weekly, 1mo for monthly
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head(50))


# IDA

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

#displays first 3, last 3, and 3 random rows of the dataframe to check the data looks like how it is expected to.
display(df.head(3))
display(df.tail(3))
display(df.sample(3))

#displays the shape of the dataframe
print("Shape of dataframe (rows, columns):", df.shape)

#summarises dataframe structure: row count, non-null counts, and data types per column
print()
df.info()

#check the number of missing values per column
print("\nMissing values per column:")
print(df.isna().sum())

#Check for duplicated rows, titles, or body text.
print()
print("Number of fully duplicated rows:", df.duplicated().any())
print("Number of duplicate titles:", df.duplicated(subset=['title']).sum())
print("Number of duplicate body text:", df.duplicated(subset=['body']).sum())

#shows summary stats for all columns
print("\nDescriptive statistics — Guardian articles:")
display(df.describe(include='all'))

#shows the count of articles by section(only the top 10 sections are shown))
print()
print("Article counts by section:")
print(df['section'].value_counts().head(10))

#confirms the actual date range of the data
print(f"\nEarliest article: {df['date'].min()}")
print(f"Latest article:   {df['date'].max()}")

#Checks word count distribution
df['word_count'] = df['body'].str.split().str.len()
print("\nWord count summary:")
print(df['word_count'].describe())
print(f"\nArticles under 50 words (likely stubs): {(df['word_count'] < 50).sum()}")


#plots article counts over time to spot unexpected gaps in coverage
monthly_counts = df.set_index('date').resample('ME').size()
fig, ax = plt.subplots(figsize=(14, 4))
monthly_counts.plot(ax=ax, color='steelblue')
ax.set_title('Monthly Article Count — Guardian Commodities Coverage (2020–2026)')
ax.set_ylabel('Number of Articles')
ax.set_xlabel('Date')
plt.tight_layout()
plt.show()


#--------------------------------------------------------------------------------------
#Now we do the same for the commodities prices data
display(df_commodities_prices.head(3))
display(df_commodities_prices.tail(3))
display(df_commodities_prices.sample(3))

print()
print("Shape:", df_commodities_prices.shape)
print("Data types:")
print(df_commodities_prices.dtypes)
print()
df_commodities_prices.info()

print()
print("Missing values per commodity:")
print(df_commodities_prices.isna().sum())

print()
print("\nAny duplicate rows?", df_commodities_prices.duplicated().any())

print()
print("\nDescriptive statistics — commodity prices:")
display(df_commodities_prices.describe())

print()
print(f"\nEarliest date: {df_commodities_prices.index.min()}")
print(f"Latest date:   {df_commodities_prices.index.max()}")

print()
print("Zero or negative price counts:")
for col in df_commodities_prices.columns:
    n = (df_commodities_prices[col] <= 0).sum()
    if n > 0:
        print(f"  {col}: {n} rows")


# Data wrangling

# EDA  

# Visualisation

rian